# CivicFlow SFT Training

Fine-tunes **Qwen/Qwen2.5-3B-Instruct** on heuristic-generated municipal-planning trajectories using Unsloth + TRL SFTTrainer.

**Setup:** Runtime → Change runtime type → T4 GPU (free tier)

**Steps:**
1. Install dependencies
2. Upload `sft_final.jsonl` from `training/sft/` when prompted
3. Run all cells — training takes ~20 min on T4
4. Download `sft_training_curve.png` and commit to `civicflow_env/assets/`

In [ ]:
# Cell 1: Install
!pip install -q unsloth trl datasets transformers accelerate bitsandbytes
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
print('Installation complete')

In [ ]:
# Cell 2: Load model with Unsloth 4-bit quantisation
from unsloth import FastLanguageModel
import torch

MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"
MAX_SEQ_LEN = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LEN,
    dtype=None,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)
print('Model loaded with LoRA adapters')

In [ ]:
# Cell 3: Upload and load SFT data
# Upload sft_final.jsonl from training/sft/ in the repo
from google.colab import files
uploaded = files.upload()  # select sft_final.jsonl

import json
from datasets import Dataset

examples = [json.loads(l) for l in open("sft_final.jsonl")]

def format_example(ex):
    msgs = ex["messages"]
    text = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)
    return {"text": text}

dataset = Dataset.from_list([format_example(e) for e in examples])
dataset = dataset.train_test_split(test_size=0.1, seed=42)
print(f"Train: {len(dataset['train'])}, Val: {len(dataset['test'])}")

In [ ]:
# Cell 4: Train
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    args=SFTConfig(
        dataset_text_field="text",
        max_seq_length=MAX_SEQ_LEN,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=10,
        num_train_epochs=3,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=10,
        eval_strategy="steps",
        eval_steps=50,
        save_strategy="steps",
        save_steps=100,
        output_dir="civicflow_sft_checkpoints",
        report_to="none",
        seed=42,
    ),
)
trainer_stats = trainer.train()
print(f"Training done. Runtime: {trainer_stats.metrics['train_runtime']:.0f}s")

In [ ]:
# Cell 5: Save adapter and push to Hub
HF_USERNAME = "Aaryan369"
REPO_ID = f"{HF_USERNAME}/civicflow-sft-qwen2.5-3b"

# Save LoRA adapter
model.save_pretrained("civicflow_sft_adapter")
tokenizer.save_pretrained("civicflow_sft_adapter")

# Merged 16-bit for inference
model.save_pretrained_merged("civicflow_sft_merged", tokenizer, save_method="merged_16bit")

# Push to Hub
model.push_to_hub(REPO_ID)
tokenizer.push_to_hub(REPO_ID)
print(f"Pushed to https://huggingface.co/{REPO_ID}")

In [ ]:
# Cell 6: Plot and save training curve
import matplotlib.pyplot as plt

log_history = trainer.state.log_history
train_loss = [(x["step"], x["loss"]) for x in log_history if "loss" in x and "eval_loss" not in x]
eval_loss  = [(x["step"], x["eval_loss"]) for x in log_history if "eval_loss" in x]

fig, ax = plt.subplots(figsize=(8, 4))
if train_loss:
    ax.plot(*zip(*train_loss), label="train loss")
if eval_loss:
    ax.plot(*zip(*eval_loss), label="eval loss", linestyle="--")
ax.set_xlabel("Training step")
ax.set_ylabel("Cross-entropy loss")
ax.set_title("CivicFlow SFT — Qwen2.5-3B-Instruct")
ax.legend()
plt.tight_layout()
plt.savefig("sft_training_curve.png", dpi=150)
plt.show()

# Download the plot — commit it to civicflow_env/assets/sft_training_curve.png
from google.colab import files
files.download("sft_training_curve.png")
print("Downloaded sft_training_curve.png — commit to civicflow_env/assets/")